In [2]:
import sys
from importlib import reload
from pathlib import Path
import pandas as pd
import isc_harmonization
from isc_harmonization import get_repo_root

TARGET_YEAR = 2024

REPO_ROOT = get_repo_root()
DATA_DIR = REPO_ROOT / "voorbeeld" / "isc_2023-2025"
MAPPING_DIR = REPO_ROOT / "mappings"

RAW_DATA_PATH = DATA_DIR / f"ISC_{TARGET_YEAR}.xlsx"

In [3]:
raw_measurements = pd.read_excel(RAW_DATA_PATH, sheet_name=str(TARGET_YEAR))
location_mapping = pd.read_excel(MAPPING_DIR / "locations-mapped.xlsx")
parameter_mapping = pd.read_excel(
    MAPPING_DIR / "parameter_mapping_final.xlsx",
    sheet_name="mapping",
)

In [6]:
raw_measurements.columns

measurements=raw_measurements[['grootheid_code',
       'parameter_code', 'hoedanigheid_code', 'compartiment_code',
       'eenheid_code']].drop_duplicates()

In [7]:
measurements


,grootheid_code,parameter_code,hoedanigheid_code,compartiment_code,eenheid_code
0,ANMLE,Gd,nf,OW,DIMSLS
13,CONCTTE,Gd,antpgnnf,OW,ug/l
26,AANWZHD,GEUR,NVT,OW,DIMSLS
45,AANWZHD,KLEUR,NVT,OW,DIMSLS
64,AANWZHD,OLE,NVT,OW,DIMSLS
...,...,...,...,...,...
29249,CONCTTE,2Ao6NO2Tol,NVT,OW,ng/l
29299,CONCTTE,34DNO2Tol,NVT,OW,ng/l
29384,CONCTTE,4Ao26DNO2Tol,NVT,OW,ng/l
29413,CONCTTE,5NO2otlidne,NVT,OW,ng/l


In [9]:
parameter_mapping
parameter_mapping['parameter_code'] = measurements['parameter_code']
parameter_mapping['grootheid_code'] = measurements['grootheid_code']
parameter_mapping['hoedanigheid_code'] = measurements['hoedanigheid_code']


,Unieke identificatie gemeten parameter,CASnummer,AQUO_Omschrijving,ISC_Parameter,parameter_code,wadar_PARCode,Unieke identificatie van de eenheid,eenheid_code,conversion,grootheid_code,hoedanigheid_code,reported
0,1082,NaN,benzo(a)antraceen,Benzo(a)antraceen,BaA,NaN,µg/L,ug/l,1.0,CONCTTE,NVT,NaN
1,1083,2921-88-2,ethylchloorpyrifos,Chloorpyrifos (Chloorpyrifos ethyl),C2yClprfs,C2yClprfs,µg/L,ug/l,1.0,CONCTTE,NVT,NaN
2,1101,15972-60-8,alachloor,Alachloor,alCl,alCl,µg/L,ug/l,1.0,CONCTTE,NVT,NaN
3,1103,309-00-2,aldrin,Aldrin,aldn,aldn,µg/L,ug/l,1.0,CONCTTE,NVT,NaN
4,1107,1912-24-9,atrazine,Atrazine,atzne,atzne,µg/L,ug/l,1.0,CONCTTE,NVT,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
125,COD,-,NaN,Opgeloste organische koolstof DOC,Corg,NaN,mg/L,mg/l,1.0,CONCTTE,Cnf,NaN
126,COT,-,NaN,Totaal organische koolstof TOC,TOC,NaN,mg/L,mg/l,1.0,CONCTTE,NVT,NaN
127,IBD,EEA_124-04-9 *,NaN,Evaluatie biologische kwaliteit - Diatomeeën,NaN,NaN,/,NaN,NaN,NaN,NaN,AM
128,FISH,EEA_14-02-8 *,NaN,Evaluatie biologische kwaliteit -Vis,NaN,NaN,/,NaN,NaN,NaN,NaN,AM


In [12]:

merge_keys = ['parameter_code', 'grootheid_code', 'hoedanigheid_code']

not_mapped = measurements.merge(
    parameter_mapping[merge_keys].drop_duplicates(),
    on=merge_keys,
    how='left',
    indicator=True
).query('_merge == "left_only"').drop(columns='_merge')

not_mapped.to_excel(DATA_DIR / f"not_mapped_{TARGET_YEAR}_RWS_data.xlsx", index=False)


In [11]:
mapped = measurements.merge(
    parameter_mapping[merge_keys].drop_duplicates(),
    on=merge_keys,
    how='inner',
)

print(f"Mapped:     {len(mapped)}")
print(f"Not mapped: {len(not_mapped)}")

mapped


Mapped:     105
Not mapped: 424


,grootheid_code,parameter_code,hoedanigheid_code,compartiment_code,eenheid_code
0,CONCTTE,123TClBen,NVT,OW,ug/l
1,CONCTTE,124TClBen,NVT,OW,ug/l
2,CONCTTE,12DClC2a,NVT,OW,ug/l
3,CONCTTE,135TClBen,NVT,OW,ug/l
4,CONCTTE,24DDT,NVT,OW,ug/l
...,...,...,...,...,...
100,T,NVT,NVT,OW,oC
101,VERZDGGD,O2,NVT,OW,%
102,pH,NVT,NVT,OW,DIMSLS
103,CONCTTE,BZV5a,NVT,OW,mg/l
